In [22]:
import pandas as pd

df = pd.read_csv('data/train.csv')
#print(df.head())
#print(df.info())
#print(df.describe())
print(df.isnull().sum())

id                                     0
Age                                    0
Flight Distance                        0
Inflight wifi service                  0
Departure/Arrival time convenient      0
Ease of Online booking                 0
Gate location                          0
Food and drink                         0
Online boarding                        0
Seat comfort                           0
Inflight entertainment                 0
On-board service                       0
Leg room service                       0
Baggage handling                       0
Checkin service                        0
Cleanliness                            0
Departure Delay in Minutes             0
Arrival Delay in Minutes             292
Gender                                 0
Customer Type                          0
Type of Travel                         0
Class                                  0
satisfaction                           0
dtype: int64


In [23]:
print(df["Arrival Delay in Minutes"].describe())
print(df["Arrival Delay in Minutes"].isnull().sum())
df["Arrival Delay in Minutes"].value_counts()

count    699343.000000
mean          1.134280
std           5.749304
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max         491.000000
Name: Arrival Delay in Minutes, dtype: float64
292


Arrival Delay in Minutes
0.0      640415
1.0        4546
2.0        4141
3.0        3881
4.0        3612
          ...  
122.0         1
120.0         1
103.0         1
227.0         1
76.0          1
Name: count, Length: 168, dtype: int64

In [20]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import KNNImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, TargetEncoder

target_col = "satisfaction"
X = df.drop(columns=["id", target_col])
y = df[target_col]

categorical_cols = X.select_dtypes(include=["object", "category"]).columns
numeric_cols = X.select_dtypes(include="number").columns

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            TargetEncoder(
                target_type="binary", smooth="auto", cv=5, random_state=42
            ),
            categorical_cols,
        ),
        (
            "numeric",
            Pipeline(
                steps=[
                    ("scaler", RobustScaler()),
                    ("imputer", KNNImputer(n_neighbors=5, weights="distance")),
                ]
            ),
            numeric_cols,
        ),
    ]
 )

preprocessing_pipeline = Pipeline(steps=[("preprocessor", preprocessor)])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
 )

X_train_processed = preprocessing_pipeline.fit_transform(X_train, y_train)
X_test_processed = preprocessing_pipeline.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)

Processed training shape: (559708, 21)
Processed test shape: (139927, 21)


In [24]:
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

X_quick_train, _, y_quick_train, _ = train_test_split(
    X_train,
    y_train,
    train_size=min(20_000, len(X_train)),
    random_state=42,
    stratify=y_train,
 )

X_quick_valid = X_test.sample(
    n=min(10_000, len(X_test)), random_state=42
 )
y_quick_valid = y_test.loc[X_quick_valid.index]

quick_model = Pipeline(
    steps=[
        ("preprocessor", clone(preprocessor)),
        (
            "classifier",
            HistGradientBoostingClassifier(
                max_iter=100,
                max_depth=6,
                learning_rate=0.1,
                l2_regularization=1.0,
                max_bins=63,
                class_weight="balanced",
                random_state=42,
            ),
        ),
    ]
 )

quick_model.fit(X_quick_train, y_quick_train)
quick_predictions = quick_model.predict(X_quick_valid)
quick_probabilities = quick_model.predict_proba(X_quick_valid)[:, 1]

print("Quick validation accuracy:", accuracy_score(y_quick_valid, quick_predictions))
print("Quick validation ROC AUC:", roc_auc_score(y_quick_valid, quick_probabilities))

Quick validation accuracy: 0.915
Quick validation ROC AUC: 0.9524125780082351


In [25]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(quick_model, X, y, scoring='roc_auc', cv=5)
print(scores)
print(scores.mean())

[0.9576167  0.95675082 0.95716166 0.95615742 0.95710649]
0.9569586179661667


In [26]:
df_test = pd.read_csv("data/test.csv")
preds = quick_model.predict_proba(df_test)[:, 1]
submission = pd.DataFrame({"id": df_test["id"], "satisfaction": preds})
submission.to_csv("data/submission.csv", index=False)